In [0]:
%run ./00_setup

In [0]:
orders_df = spark.table(f"workspace.bronze.orders")
customers_keys = spark.table(f"workspace.bronze.customers").select("customer_id").distinct()

email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
dup_w = Window.partitionBy("order_id")

flagged = (
    orders_df
    .withColumn("expected_total", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("dup_count", F.count("*").over(dup_w))
    .join(
        customers_keys.withColumnRenamed("customer_id", "customer_id_ref"),
        orders_df.customer_id == F.col("customer_id_ref"),
        "left"
    )
    .withColumn("rule_order_id_not_null", F.col("order_id").isNotNull())
    .withColumn("rule_order_id_unique", F.col("dup_count") == 1)
    .withColumn("rule_customer_id_not_null", F.col("customer_id").isNotNull())
    .withColumn("rule_customer_exists", F.col("customer_id_ref").isNotNull())
    .withColumn("rule_country_valid", F.col("country").isin("ES", "MX", "AR", "CO", "CL"))
    .withColumn("rule_currency_valid", F.col("currency").isin("EUR", "MXN", "ARS", "COP", "CLP"))
    .withColumn(
        "rule_currency_matches_country",
        ((F.col("country") == "ES") & (F.col("currency") == "EUR")) |
        ((F.col("country") == "MX") & (F.col("currency") == "MXN")) |
        ((F.col("country") == "AR") & (F.col("currency") == "ARS")) |
        ((F.col("country") == "CO") & (F.col("currency") == "COP")) |
        ((F.col("country") == "CL") & (F.col("currency") == "CLP"))
    )
    .withColumn("rule_quantity_range", F.col("quantity").between(1, 100))
    .withColumn("rule_unit_price_range", F.col("unit_price").between(0.01, 10000))
    .withColumn("rule_total_matches", F.abs(F.col("order_total") - F.col("expected_total")) < F.lit(0.01))
    .withColumn("rule_payment_method_valid", F.col("payment_method").isin("CARD", "TRANSFER", "CASH", "PAYPAL"))
    .withColumn("rule_order_status_valid", F.col("order_status").isin("CREATED", "PAID", "CANCELLED", "REFUNDED"))
    .withColumn("rule_email_valid", F.col("email").rlike(email_regex))
    .withColumn("rule_order_ts_not_future", F.col("order_ts") <= F.current_timestamp())
)

In [0]:
flagged.display()

In [0]:
import pyspark.sql.functions as F

rule_cols = [
    "rule_order_id_not_null", "rule_order_id_unique", "rule_customer_id_not_null",
    "rule_customer_exists", "rule_country_valid", "rule_currency_valid",
    "rule_currency_matches_country", "rule_quantity_range", "rule_unit_price_range",
    "rule_total_matches", "rule_payment_method_valid", "rule_order_status_valid",
    "rule_email_valid", "rule_order_ts_not_future"
]

# 1. Creamos el arreglo con los nombres de las reglas que fallaron (y nulos para las que pasaron)
# Usar (F.col(c) == False) es más seguro que ~F.col(c) por si llegan a haber valores NULL.
array_with_nulls = F.array(*[
    F.when(F.col(c) == False, F.lit(c)) for c in rule_cols
])

flagged = (
    flagged
    .withColumn(
        "failed_rules",
        # 2. Usamos F.filter para quedarnos solo con los elementos que NO son nulos
        F.filter(array_with_nulls, lambda x: x.isNotNull())
    )
    .withColumn(
        "is_valid", 
        # 3. Si el tamaño del arreglo es 0, no hay reglas fallidas (es válido)
        F.size(F.col("failed_rules")) == 0
    )
)

display(flagged.select("order_id", "customer_id", "failed_rules", "is_valid").limit(5))

In [0]:
orders_silver = (
    flagged
    .filter(F.col("is_valid"))
    .drop("expected_total", "dup_count", "customer_id_ref", *rule_cols, "failed_rules", "is_valid")
)

orders_quarantine = (
    flagged
    .filter(~F.col("is_valid"))
)

orders_silver.write.mode("overwrite").format("delta").saveAsTable(f"workspace.silver.orders_silver")
orders_quarantine.write.mode("overwrite").format("delta").saveAsTable(f"workspace.silver.orders_quarantine")

In [0]:
orders_silver.display()

In [0]:
orders_quarantine.display()

In [0]:
raw_count = spark.table(f"workspace.bronze.orders").count()
silver_count = spark.table(f"workspace.silver.orders_silver").count()
quarantine_count = spark.table(f"workspace.silver.orders_quarantine").count()

metrics = spark.createDataFrame(
    [(raw_count, silver_count, quarantine_count)],
    ["raw_count", "silver_count", "quarantine_count"]
)

display(metrics)